# 3.02 - Visualizing Airborne Events

Plotting top 10 most haunted airplane routes and 10 most haunted airports for each type. 
Possible types include: 
- **small_airports** [Class D] 
    - 3 Nautical Miles 
    - Any airport with a control tower
- **medium_Airport** [Class C] 
    - 5 Nautical Miles
    - "Airports of Moderate Importance"
- **large_airport**  [Class B] 
    - 30 Nautical Miles
    - Large Commercial Airports
- **Heliports**
    - 1.5 Nautical Miles
    - Figure 7-1 in [Heliport_Guidelines](https://www.faa.gov/documentLibrary/media/Advisory_Circular/AC_150_5390_2D_Heliports.pdf) requires heliports to have a minimum airspace of 4,000 ft. We assumed areas within 3 miles of a heliport would be visible.
- **balloonports, seaplane_base**
    - 3 Nautical Miles


**All Flight and airport data data is from notebooks [2.01, 3.00]**



In [2]:
# System Path #
import os
import sys 

# Add dsci_550_a1 to base path. Lets you project functions #
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Pandas #
import pandas as pd
import time
import re
import json 


# Runtime #
import time
from tqdm import tqdm 

# Iterators #
import collections
import ast
import random

# Flight Trajectory Functions #
from dsci_550_a1.flightFunctions import *

# Plotting #
import plotly.graph_objects as go


## Load haunted places with added features
df_american_routes = pd.read_csv("../data/joined_datasets/american_routes.tsv", sep = "\t")

## Our Airports
df_american_airports = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep = "\t")

## Open Flights Dataset
df_haunted_places = pd.read_csv("../data/processed/haunted_places_features_added.tab", sep = "\t")

## Pandas stores nested dicts and lists as strings ##
## This converts them back to lists and dicst ##

df_american_routes["Flight_Path"] = df_american_routes["Flight_Path"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_american_airports["Airport_Radius"] = df_american_airports["Airport_Radius"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

## Flight Intersection Data
with open("../data/processed/flight_proximity_data.json") as f:
    flight_intersection_data = json.load(f)
    
## Airport Intersection Data
with open("../data/processed/airport_proximity_data.json") as f:
    airport_intersection_data = json.load(f)


## Number of Haunted Places each flight intersected
with open("../data/processed/flight_haunted_place_counts.json") as f:
    flight_haunt_counts = json.load(f)
    
## Number of haunted places each airport intersected
with open("../data/processed/airport_haunted_place_counts.json") as f:
    airport_haunt_counts = json.load(f)

In [4]:

col = 'Apparition_Type'
flagged_values = pd.unique(df_haunted_places['Apparition_Type']).tolist()
res = []
for entry in flagged_values:
    entries = list(map(lambda x: x.strip(), entry.split(',')))
    [res.append(entry) for entry in entries if entry not in res]

flag_ranks = {}
for flag in res:
    flag_ranks[flag] = 0
    for key in  df_haunted_places[col].value_counts().keys():
        if flag in key:
            flag_ranks[flag] += df_haunted_places[col].value_counts()[key]

flag_ranks = sorted(flag_ranks.keys(), key = lambda x: flag_ranks[x], reverse = False)


for flag in flag_ranks:
    for idx in df_haunted_places.index:
        if key in df_haunted_places.loc[idx, col] == flag:
            df_haunted_places[col] = flag


In [5]:
####################################################################################################
## Flag relevant values and specify column
col = 'Apparition_Type'
flagged_values = pd.unique(df_haunted_places[f'{col}']).tolist()
res = []
for entry in flagged_values:
    entries = list(map(lambda x: x.strip(), entry.split(',')))
    [res.append(entry) for entry in entries if entry not in res]

#############################################
## Assign Flagged values in hierarchy
# plotly does not allow for multi-legend entries. Because of this each haunted place gets a single legend name. 
# For entries that have more than one flagged value, we assign it whatever flag has the least amount of total counts in the dataframe.
# ! Rarest values have priority !

flag_ranks = {}
for flag in res:
    flag_ranks[flag] = 0
    for key in  df_haunted_places[col].value_counts().keys():
        if flag in key:
            flag_ranks[flag] += df_haunted_places[col].value_counts()[key]

flag_ranks = sorted(flag_ranks.keys(), key = lambda x: flag_ranks[x], reverse = False)


for flag in flag_ranks:
    for idx in df_haunted_places.index:
        if key in df_haunted_places.loc[idx, col] == flag:
            df_haunted_places[col] = flag

## Filter Df
df_haunted_places_filtered = df_haunted_places[df_haunted_places[col].apply(lambda x: any(event in x for event in flagged_values))].drop_duplicates()

## Store entry IDs
# IDs are used to link flight and airport intersection data to haunted place
haunted_places_filtered_idxs = df_haunted_places_filtered.index.tolist()


####################################################################################################
## Filter df_american_airports and df_american_routes


## Init lists to store flight route and airport 

route_idxs = sorted(flight_haunt_counts.keys(), key = lambda x : flight_haunt_counts[x], reverse = True)[:10]
airport_idxs =sorted(airport_haunt_counts.keys(), key = lambda x : airport_haunt_counts[x], reverse = True)[:10]
airport_Iata_codes = []
airport_idxs =[]


## Filter routes df

routes_filtered = pd.DataFrame(columns=df_american_routes.columns)
x = 0 
for key, val  in flight_haunt_counts.items():
    for idx in df_american_routes.index:
        source, dest = df_american_routes.loc[idx, ['Source_Airport', 'Destination_Airport']]
        if key == f"{source} -> {dest}":
            routes_filtered = pd.concat([routes_filtered, df_american_routes.loc[[idx]]], ignore_index=True, axis = 0)
            routes_filtered.loc[x, 'Intersection_Count'] = int(val)
            routes_filtered.loc[x, 'Name'] = key
            x += 1
            break
    if x == 10:
        break


## Filter Airports df
idxs = []
# for airport_type in airport_haunt_counts:
#     x = 0
#     for id, count in airport_haunt_counts[airport_type]:
#         idxs.append(id)
#         x += 1
#         if x == 10:
#             break
        
# airports_filtered = df_american_airports.loc[idxs]

## Number of haunted places each airport intersected
# with open("../data/processed/airport_haunted_place_counts_names.json") as f:
#     airport_haunt_counts = json.load(f)

airports_filtered = pd.DataFrame(columns=df_american_airports.columns)
for airport_type in airport_haunt_counts:
    x = 0
    names = []
    for name, count in airport_haunt_counts[airport_type]:
        names.append(name)
        x += 1
        if x == 10:
            break
    airports_filtered = pd.concat([airports_filtered, df_american_airports.loc[df_american_airports['Name'].isin(names)]])
    


/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_94392/2930910695.py:59: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  routes_filtered = pd.concat([routes_filtered, df_american_routes.loc[[idx]]], ignore_index=True, axis = 0)
/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_94392/2930910695.py:93: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  airports_filtered = pd.concat([airports_filtered, df_american_airports.loc[df_american_airports['Name'].isin(names)]])


In [7]:
import textwrap

##########################################################################################
## Haunted Places Traces ##

# Initialize lists to store traces
all_traces = []
haunting_traces_idx = [] 
airports_traces_idx = [] 
flights_traces_idx = []


unique_flags = df_haunted_places_filtered[f'{col}'].unique().tolist()

plot_colors = ['rgb(102,179,92)',
 'rgb(14,106,71)',
 'rgb(188,20,102)',
 'rgb(121,210,214)',
 'rgb(74,202,87)',
 'rgb(116,99,103)',
 'rgb(151,130,149)',
 'rgb(52,1,87)',
 'rgb(235,157,37)',
 'rgb(129,191,187)',
 'rgb(20,160,203)',
 'rgb(57,21,252)',
 'rgb(235,88,48)',
 'rgb(218,58,254)',
 'rgb(169,255,219)',
 'rgb(187,207,14)',
 'rgb(189,189,174)']
# plot_colors = ['rgb{240,163,255}','rgb{0,117,220}','rgb{153,63,0}','rgb{76,0,92}','rgb{25,25,25}','rgb{0,92,49}','rgb{43,206,72}','rgb{255,204,153}','rgb{128,128,128}','rgb{148,255,181}','rgb{143,124,0}','rgb{157,204,0}','rgb{194,0,136}','rgb{0,51,128}','rgb{255,164,5}','rgb{255,168,187}','rgb{66,102,0}','rgb{255,0,16}','rgb{94,241,242}','rgb{0,153,143}','rgb{224,255,102}','rgb{116,10,255}','rgb{153,0,0}','rgb{255,255,128}','rgb{255,255,0}','rgb{255,80,5}']

## Add Trace Depending on Haunted Type ##
for i, flag in enumerate(flag_ranks):
    haunted_places_to_plot = df_haunted_places_filtered.loc[df_haunted_places_filtered[f'{col}'] == flag]
    haunted_places_to_plot['Formatted_Description'] = haunted_places_to_plot['Description'].apply(
    lambda x: "<br>".join(textwrap.wrap(x, width=50))
)
    trace = (go.Scattergeo(
        locationmode = 'USA-states',
        lon = haunted_places_to_plot['Longitude'],
        lat = haunted_places_to_plot['Latitude'],
        hoverinfo = 'text',
        text = haunted_places_to_plot.apply(lambda row: f"Haunting Type: {row[f'{col}']}<br>index:{row['Haunted_Places_Id']} | Location: {row['Location']}<br># Intersecting Flights: {row['Flight_Intersection_Count']} | # Nearby Airports: {row['Aerodrome_Count']}<br>Description: {row['Formatted_Description']}", axis=1),
        mode = 'markers',
        showlegend = True, 
        marker = dict(
            size = 4,
            color = plot_colors[i],
            opacity = 0.75
            ),
            name = flag,
            visible = False
        )
    )
    # Add trace
    all_traces.append(trace)
    # Store index of trace in haunting_traces_idx
    haunting_traces_idx.append(len(all_traces) - 1)


## Unpack Flight Path Coords ##
lats_plot, lons_plot = [] , []
for row in routes_filtered.itertuples(index = False):   

    lats, lons = zip(*row.Flight_Path)
    lats, lons = list(lats), list(lons)

    lats_plot.extend(lats + [None])
    lons_plot.extend(lons + [None])

## Add Flight Path Trace ##
trace = (go.Scattergeo(
    lon= lons_plot,
    lat= lats_plot,
    mode='lines',
    line=dict(width=1.5, color='red'),
    opacity = 0.75, 
    hoverinfo = 'text', 
    text = routes_filtered.apply(lambda row: f"Name: {row['Name']}<br>Intersection Count:{row['Intersection_Count']}" , axis=1),
    name = "Flights",
    visible = False
))
# Add Flight Trace to all traces
all_traces.append(trace)
# Store index of flight trace in flight_traces_idx
flights_traces_idx.append(len(all_traces) - 1)

## Add Airports ##

airport_types = airports_filtered['Type'].unique().tolist()

airport_plot_colors = {
'heliport' :        "rgb(100,151,177)" ,
 'seaplane_base': 	"rgb(179,205,224)",
 'balloonport' : 	"rgb(179,205,224)",
 'small_airport' :  "rgb(0,91,150)"  ,
 'medium_airport' :	"rgb(3,57,108)",
 'large_airport':   "rgb(1,31,75)"
}

airport_proximity_dict = {
    "large_airport" : 55560,    # 30 nautical miles
    "medium_airport" : 9260,    # 5 nautical miles
    "small_airport" : 5556,     # 3 nautical miles
    "heliport":  2778,          # 1.5 nautical miles
    "seaplane_base" : 5556,     # 3 nautical miles
    "balloonport" : 5556        # 3 nautical miles
}

## Plot airports ##
for airport_type in airport_types:

    ## Airport Marker Trace ##
    airports_to_plot = airports_filtered.loc[airports_filtered['Type'] == airport_type]

    airports_trace = (go.Scattergeo(
    locationmode = 'USA-states',
    lon = airports_to_plot['Longitude_Deg'],
    lat = airports_to_plot['Latitude_Deg'],
    hoverinfo = 'text',
    text = airports_to_plot.apply(lambda row: f"IATA Code: {row['Iata_Code']}<br>Name: {row['Name']}", axis=1),
    # text = airports_to_plot['Iata_Code'],
    mode = 'markers',
    marker = dict(
        size = 4,
        color = airport_plot_colors[airport_type],
        opacity = 1
        ),
        name = airport_type,
        visible = False
        ))
    
    ## Airport Radius Trace ##
    lons_plot = [] 
    lats_plot = []
    for airport in airports_to_plot.itertuples():
        ## Unpack Coords ##
        lats, lons = zip(*airport.Airport_Radius)
        lats, lons = list(lats), list(lons)

        lats_plot.extend(lats + [None])
        lons_plot.extend(lons + [None])
    
    airport_radii = (go.Scattergeo(
    locationmode = 'USA-states',
    lon = lons_plot,
    lat = lats_plot,
    hoverinfo = 'skip',
    mode = 'lines',
    line = dict(
        width = 1,
        color = airport_plot_colors[airport_type],
        dash = 'dot'
        ),
        name = airport_type,
        visible = False
        ))
    
    # Add airport trace to plot
    all_traces.append(airports_trace)
    # Add airport radius to plot
    all_traces.append(airport_radii)
    # Store index of aiport and airport radius in airport_traces_idx
    airports_traces_idx.append((len(all_traces) - 2 , len(all_traces) - 1))



##########################################################################################
## Menus ##

## Add Interactive Buttons for Haunts ##

button_haunts = [
        {
            "method": "restyle",

            "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name == flagged_value]],
            # When toggled off, checkbox removes haunted trace
            "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name == flagged_value]],
            # "args2" : ["visible", [trace.visible for trace in fig.data]], 
            "label": flagged_value,
            "visible" : True, 

        }
        for flagged_value in flag_ranks
    ]

all_haunts_button = {
            "method": "restyle",
            # When toggled on, checkbox shows already visible traces + haunted place specified in box
            "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name in flag_ranks]],
            # When toggled off, checkbox removes haunted trace
            "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name in flag_ranks]],
            # "args2" : ["visible", [trace.visible for trace in fig.data]], 
            "label": "Toggle All",
            "visible" : True, 

        }
button_haunts.append(all_haunts_button) 


## Add Interactive Buttons for airports ##

buttons_airports = [
        {
            "method": "restyle",
            # When toggled on, checkbox shows already visible traces + haunted place specified in box
            "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name == airport_type]],
            # When toggled off, haunted trace is removed from graph. Only legend is visible
            "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name == airport_type]],

            "label": airport_type,
            "visible" : True, 
        }
        for airport_type in airport_types
    ]

## Show all airports button ##
all_airports_button = {
            "method": "restyle",
            # When toggled on, checkbox shows already visible traces + haunted place specified in box
            "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name in airport_types]],
            # When toggled off, checkbox removes haunted trace
            "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name in airport_types]],
            # "args2" : ["visible", [trace.visible for trace in fig.data]], 
            "label": "Toggle All",
            "visible" : True, 

        }
buttons_airports.append(all_airports_button) 

## Show all flights button ##
flight_button = {
            "method": "restyle",
            # When toggled on, checkbox shows already visible traces + haunted place specified in box
            "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name == "Flights"]],
            # When toggled off, checkbox removes haunted trace
            "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name == "Flights"]],
            # "args2" : ["visible", [trace.visible for trace in fig.data]], 
            "label": "Flight Paths",
            "visible" : True, 

        }
buttons_airports.append(flight_button)



## Final Conf ##

updateMenusConf = [
    {
        "buttons": button_haunts,
        "direction" : "down",
        "showactive" : False,
        "x": 0.1,
        "y": 1.15,
        "xanchor" : "left",
        "yanchor" : "top", 
        "font": {"size" : 12},
        "type": "dropdown",
        "name": "Toggle Flags"
    },
    {
        "buttons": buttons_airports,
        "direction" : "down",
        "showactive" : True,
        "x": -.05,
        "y": 1.15,
        "xanchor" : "left",
        "yanchor" : "top", 
        "font": {"size" : 12},
        "type": "dropdown",
        "name": "Flight Toggle"
    }
    ]


## Create Plotly figure ##
fig = go.Figure(data = all_traces)


fig.update_layout(
    title_text = 'Flight Paths Accross U.S.',
    showlegend = True,
    clickmode='event+select',
    hovermode = 'closest',
    geo = dict(
        scope = 'north america',
        projection_type = 'azimuthal equal area',
        showland = True,
        showcountries = True,
        showsubunits = True, 
        subunitcolor = "Black",
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(204, 204, 204)',
    ),
    updatemenus = updateMenusConf
)

# Remove html if it already exists 
if os.path.exists('mostHauntedAirports.html'):
    os.remove("mostHauntedAirports.html")

fig.write_html("mostHauntedAirports.html")


# Show plot
fig.show()



/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_94392/3023666509.py:37: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_94392/3023666509.py:37: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_94392/3023666509.py:37: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th